# Summarize Evaluation Results

In [ ]:
import json
from pathlib import Path

import pandas as pd

OUTPUT_DIR = Path("../outputs/evaluation")

RUNS = {
    "Base model + persona": (
        OUTPUT_DIR
        / "qwen25_05b_base_5000_eval"
        / "summary.json"
    ),
    "Fine-tuned model + persona": (
        OUTPUT_DIR
        / "qwen25_05b_compact_5000_eval"
        / "summary.json"
    ),
    "Base model, no persona": (
        OUTPUT_DIR
        / "qwen25_05b_no_persona_base_5000"
        / "summary.json"
    ),
    "Fine-tuned model, no persona": (
        OUTPUT_DIR
        / "qwen25_05b_no_persona_5000"
        / "summary.json"
    ),
}

In [ ]:
def load_evaluation_summary(
    model_name: str,
    summary_path: Path,
) -> dict:
    """Load the main metrics from one evaluation summary."""

    if not summary_path.exists():
        raise FileNotFoundError(
            f"Evaluation summary not found: {summary_path}"
        )

    with summary_path.open(
        encoding="utf-8"
    ) as file:
        summary = json.load(file)

    return {
        "model": model_name,
        "model_type": summary["model_type"],
        "examples": summary["examples"],
        "valid_json_pct": (
            100 * summary["valid_json_rate"]
        ),
        "format_repair_pct": (
            100 * summary["format_repair_rate"]
        ),
        "valid_schema_pct": (
            100 * summary["valid_schema_rate"]
        ),
        "scoreable_response_pct": (
            100 * summary["scoreable_response_rate"]
        ),
        "exact_match_pct": (
            100 * summary["exact_match_accuracy"]
        ),
        "normalized_accuracy_pct": (
            100 * summary["normalized_accuracy"]
        ),
        "task_weighted_accuracy_pct": (
            100
            * summary[
                "task_weighted_normalized_accuracy"
            ]
        ),
        "task_weighted_accuracy_including_invalid_pct": (
            100
            * summary[
                "task_weighted_normalized_accuracy_including_invalid"
            ]
        ),
        "scored_responses": summary[
            "scored_responses"
        ],
        "eligible_responses": summary[
            "eligible_responses"
        ],
        "scored_tasks": summary["scored_tasks"],
    }

In [ ]:
model_results = pd.DataFrame(
    [
        load_evaluation_summary(
            model_name,
            summary_path,
        )
        for model_name, summary_path in RUNS.items()
    ]
).set_index("model")

model_results.round(2)

In [ ]:
report_results = model_results[
    [
        "examples",
        "valid_json_pct",
        "valid_schema_pct",
        "scoreable_response_pct",
        "exact_match_pct",
        "task_weighted_accuracy_including_invalid_pct",
    ]
].rename(
    columns={
        "examples": "Examples",
        "valid_json_pct": "Valid JSON (%)",
        "valid_schema_pct": "Valid schema (%)",
        "scoreable_response_pct": (
            "Scoreable responses (%)"
        ),
        "exact_match_pct": "Exact match (%)",
        "task_weighted_accuracy_including_invalid_pct": (
            "Task-weighted accuracy (%)"
        ),
    }
)

report_results.round(2)